# AI-Lake + Trino

Trino reads AI-Lake tables via the standard **Iceberg connector** —
no AI-Lake plugin required for tabular queries.

**Prerequisites:** start the engines compose overlay:
```bash
docker compose \
  -f tests/docker/compose-demo.yml \
  --profile engines \
  up -d
```

**What this notebook covers:**
1. Connect to Trino via the Python `trino` driver
2. Discover catalogs, schemas, tables
3. SQL queries: scan, filter, aggregation
4. `embedding` column as `varbinary`
5. Iceberg metadata queries (`$snapshots`, `$manifests`)

In [ ]:
import os
import trino

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

print(f'Connecting to Trino at {TRINO_HOST}:{TRINO_PORT} ...')

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user='ailake',
    catalog='ailake',
    schema='default',
)
cur = conn.cursor()

def q(sql):
    """Execute SQL and return rows."""
    cur.execute(sql)
    return cur.fetchall()

def qdf(sql):
    """Execute SQL and return a pandas DataFrame."""
    import pandas as pd
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

print('Connected.')

In [ ]:
# ── Pre-flight check — verify Trino is reachable ─────────────────────────────
import os, socket

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

try:
    with socket.create_connection((TRINO_HOST, TRINO_PORT), timeout=5):
        print(f"OK: Trino reachable at {TRINO_HOST}:{TRINO_PORT}")
except OSError:
    raise RuntimeError(
        f"\n\nTrino is not reachable at {TRINO_HOST}:{TRINO_PORT}.\n"
        "Start the engines overlay first:\n\n"
        "  docker compose \\\n"
        "    -f tests/docker/compose-demo.yml \\\n"
        "    -f tests/docker/compose-demo-engines.yml \\\n"
        "    up -d\n"
    )

## 1. Discover catalogs and tables

In [ ]:
print('Catalogs:', q('SHOW CATALOGS'))
print('Schemas :', q('SHOW SCHEMAS FROM ailake'))
print('Tables  :', q('SHOW TABLES FROM ailake.default'))

## 2. Schema inspection

In [ ]:
# embedding is varbinary — standard Parquet FIXED_LEN_BYTE_ARRAY read as bytes
qdf('DESCRIBE ailake.default."table"')

## 3. Basic scan

In [ ]:
count = q('SELECT count(*) FROM ailake.default."table"')[0][0]
print(f'Total rows: {count}')

In [ ]:
qdf("""
    SELECT text,
           length(embedding) AS embedding_bytes
    FROM ailake.default."table"
    LIMIT 5
""")

## 4. Filtered query + aggregation

In [ ]:
df = qdf("""
    SELECT text
    FROM ailake.default."table"
    WHERE text LIKE '%vector search%'
""")
print(f"Rows about 'vector search': {len(df)}")
df.head(3)

In [ ]:
qdf("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_chars,
        avg(length(embedding)) AS avg_embedding_bytes
    FROM ailake.default."table"
""")

## 5. Iceberg metadata — snapshots and manifests

In [ ]:
# Trino Iceberg connector exposes system tables via the $ suffix
qdf('SELECT snapshot_id, committed_at, operation FROM "ailake"."default"."table$snapshots"')

In [ ]:
qdf("""
    SELECT
        file_path,
        record_count,
        file_size_in_bytes
    FROM "ailake"."default"."table$files"
""")

## Key takeaway

Trino queries AI-Lake tables through the **standard Iceberg connector** —
the Parquet files, manifest Avro files, and `metadata.json` all conform to
Iceberg Spec v2 and Trino reads them without any modification.

The HNSW footer section is **physically after** the Parquet `PAR1` magic bytes;
Trino's Parquet reader stops at `PAR1` and never sees it.

To enable vector search from Trino, install the AI-Lake Trino plugin
(see `docs/specs/JVM_PLUGINS.md`).